<a href="https://colab.research.google.com/github/AdityaR28/OSS4008A---Introduction-to-Remote-Sensing-and-Applications/blob/main/students/devon_clermont-gauer/code/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# Water Dashboard (Project 11)
#**Name:** Devon Clermont-Gauer

#Interactive map of water bodies in Canada. Reference layers (all, permanent and seasonal water from JRC; rivers from HydroSHEDS) can be toggled with the buttons, and **AI water detection** runs the trained U-Net on Sentinel-2 imagery inside a box the user draws. The chat panel is a placeholder for a future AI assistant.

#**How to run:**
#1. Replace `YOUR-PROJECT-ID` in Cell 1 with your Google Earth Engine Cloud project ID.
#2. Make sure the trained model and band statistics are in your Google Drive (paths in Cell 4).
#3. Runtime → Run all (a T4 GPU runtime makes AI detection faster, but CPU works).

In [16]:
# Cell 1: install libraries and log in to Earth Engine
!pip install -q geemap segmentation-models-pytorch

import ee
import geemap

ee.Authenticate()
ee.Initialize(project='causal-sky-509317-u9')                    # replace with your Cloud project ID
print('Earth Engine says:', ee.String('hello').getInfo())   # connection test

Earth Engine says: hello


In [17]:
# Cell 2: download this project's code from the course repo (public, no login needed)
import os, sys, subprocess

REPO     = 'OSS4008A---Introduction-to-Remote-Sensing-and-Applications'
REPO_DIR = f'/content/{REPO}'
CODE_DIR = f'{REPO_DIR}/students/devon_clermont-gauer/code'         # replace YOUR-FOLDER with your folder name

if os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)           # update to latest
else:
    subprocess.run(['git', 'clone', '--depth', '1',
                    f'https://github.com/AdityaR28/{REPO}.git', REPO_DIR], check=True)

sys.path.insert(0, CODE_DIR)                                 # let Python find the modules
for f in ['water_layers.py', 'water_ai.py', 'band_stats.json']:
    print(f, 'found' if os.path.exists(f'{CODE_DIR}/{f}') else 'MISSING')

water_layers.py found
water_ai.py found
band_stats.json found


In [18]:
# Cell 3: download the trained model (shared on Google Drive; too large for the repo)
import gdown

MODEL_ID   = '1ZvIHmtM5tdOvZHpJC5JdmKNEVgkQbkL4'    # from the model's Drive share link
MODEL_PATH = '/content/unet_resnet34_best.pth'
STATS_PATH = f'{CODE_DIR}/band_stats.json'          # band statistics, stored in the repo

if not os.path.exists(MODEL_PATH):
    gdown.download(id=MODEL_ID, output=MODEL_PATH, quiet=False)
print('Model ready:', os.path.exists(MODEL_PATH))

Downloading...
From (original): https://drive.google.com/uc?id=1ZvIHmtM5tdOvZHpJC5JdmKNEVgkQbkL4
From (redirected): https://drive.google.com/uc?id=1ZvIHmtM5tdOvZHpJC5JdmKNEVgkQbkL4&confirm=t&uuid=d2b8287e-a6c8-4ab7-b881-26c137da58f2
To: /content/unet_resnet34_best.pth
100%|██████████| 97.9M/97.9M [00:03<00:00, 28.7MB/s]

Model ready: True


In [19]:
# Cell 4: dashboard UI (chat panel | map, project buttons underneath)
import ipywidgets as widgets
from water_layers import (get_overview_layer, get_permanent_layer,
                          get_seasonal_layer, get_rivers_layer, canada)
from water_ai import load_model, detect_water_ai, water_overlay

# Load the trained U-Net (downloaded in Cell 3) and the band statistics (from the repo)
load_model(MODEL_PATH, STATS_PATH)

# --- Styling (Google-like: Roboto, cards, pill buttons, chat bubbles) ---
css = widgets.HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Roboto:wght@400;500&display=swap');
.app       {font-family: Roboto, Arial, sans-serif; background:#f8f9fa;
            border-radius:16px; padding:16px;}
.title     {font-size:22px; font-weight:500; color:#202124; margin:0 0 12px 4px;}
.card      {background:#fff; border-radius:12px; padding:12px;
            box-shadow:0 1px 3px rgba(60,64,67,.3), 0 4px 8px rgba(60,64,67,.15);}
.map-card  {border-radius:12px; overflow:hidden; padding:0;
            box-shadow:0 1px 3px rgba(60,64,67,.3), 0 4px 8px rgba(60,64,67,.15);}
.chat-log  {height:480px; overflow-y:auto; display:flex; flex-direction:column-reverse;}
.chat-log .widget-html-content {display:flex; flex-direction:column; margin-bottom:auto;}
.bot  {background:#e8f0fe; color:#202124; border-radius:16px 16px 16px 4px;
       padding:8px 12px; margin:4px 0; max-width:85%; width:fit-content;}
.user {background:#1a73e8; color:#fff; border-radius:16px 16px 4px 16px;
       padding:8px 12px; margin:4px 0 4px auto; max-width:85%; width:fit-content;}
.pill.widget-button {border-radius:20px !important; background:#fff !important;
       color:#1a73e8 !important; border:1px solid #dadce0 !important;
       font-weight:500; height:36px; padding:0 18px; margin-right:8px;}
.pill.widget-button:hover {background:#f1f3f4 !important;}
.pill.active.widget-button {background:#1a73e8 !important; color:#fff !important;
       border-color:#1a73e8 !important;}
.chat-input input {border-radius:20px !important; border:1px solid #dadce0 !important;
       padding:0 14px !important; height:36px;}
.leaflet-control-attribution {font-size:9px !important; opacity:0.5;
       background:transparent !important; line-height:1.2;}
.leaflet-control-attribution:hover {opacity:1;}
</style>""")

def bubble(text, who):
    """Wrap a message in a chat bubble ('bot' = assistant, 'user' = user)."""
    return f'<div class="{who}">{text}</div>'

# --- Map ---
ALL_LAYERS = [get_overview_layer, get_permanent_layer,
              get_seasonal_layer, get_rivers_layer]

def build_map(layer_fn=None):
    """Create a fresh map of Canada.
    layer_fn: None = base map, a layer function = that layer only,
              'all' = every layer, toggled in the map's layer control."""
    new_map = geemap.Map(center=[60, -95], zoom=4, height='560px',
                         basemap='Esri.WorldImagery')   # Esri (OpenStreetMap tiles are blocked in Colab)
    new_map.addLayer(canada.style(color='ffffff', fillColor='00000000', width=1),
                     {}, 'Canada boundary')

    fns = ALL_LAYERS if layer_fn == 'all' else ([layer_fn] if layer_fn else [])
    legend = {}
    for i, fn in enumerate(fns):
        img, vis, name = fn()
        new_map.addLayer(img, vis, name, i == 0)        # first layer visible, others ticked off
        legend[name] = '#' + vis.get('palette', ['00e5ff'])[0]
    if legend:
        new_map.add_legend(title='Layers', legend_dict=legend)

    new_map.layout.width = '100%'
    return new_map

# Container so the map can be swapped out when a button reloads it
map_area = widgets.Box([build_map()], layout=widgets.Layout(width='70%', margin='0 0 0 12px'))
map_area.add_class('map-card')

# --- Chat panel (placeholder until an AI assistant is connected) ---
chat_log = widgets.HTML(
    bubble('Hi! Draw a box on the map and click "AI: water in drawn box" '
           'to detect water with the trained model.', 'bot')
    + bubble('Base map has loaded.', 'bot'))
chat_log.add_class('chat-log')

chat_in = widgets.Text(placeholder='Ask about water in Canada...',
                       layout=widgets.Layout(width='75%'))
chat_in.add_class('chat-input')
send_btn = widgets.Button(description='Send')
send_btn.add_class('pill')

def handle_message(_):
    """Show the user's message and a placeholder reply."""
    if chat_in.value.strip():
        chat_log.value += bubble(chat_in.value, 'user')
        chat_log.value += bubble('The chat assistant is not connected yet.', 'bot')
        chat_in.value = ''

send_btn.on_click(handle_message)
chat_panel = widgets.VBox([widgets.HTML('<div class="title">Assistant</div>'),
                           chat_log, widgets.HBox([chat_in, send_btn])],
                          layout=widgets.Layout(width='30%'))
chat_panel.add_class('card')

# --- Reference layer buttons: (label, icon, what to load) ---
layer_buttons = [
    ('Base map',        'globe',       None),
    ('All water',       'tint',        get_overview_layer),
    ('Permanent water', 'tint',        get_permanent_layer),
    ('Seasonal water',  'leaf',        get_seasonal_layer),
    ('Rivers',          'road',        get_rivers_layer),
    ('All layers',      'layer-group', 'all'),
]

btns = []

def make_handler(btn, label, layer_fn):
    """Build a click handler that reloads the map with this button's layer(s)."""
    def handler(_):
        for b in btns:
            b.remove_class('active')
        btn.add_class('active')
        chat_log.value += bubble(f'Loading {label.lower()}...', 'bot')
        try:
            map_area.children = (build_map(layer_fn),)
            chat_log.value += bubble(f'{label} has loaded.', 'bot')
        except Exception as e:
            chat_log.value += bubble(f'Could not load {label.lower()}: {e}', 'bot')
    return handler

for label, icon, fn in layer_buttons:
    b = widgets.Button(description=label, icon=icon)
    b.add_class('pill')
    b.on_click(make_handler(b, label, fn))
    btns.append(b)
btns[0].add_class('active')

# --- Draw-a-box AI analysis: U-Net water detection inside a user-drawn rectangle ---
def water_in_box(_):
    m = map_area.children[0]                    # the map currently shown
    box = m.user_roi                            # last shape drawn with the map's draw tools
    if box is None:
        chat_log.value += bubble('Draw a rectangle on the map first (square tool on the left).', 'bot')
        return

    chat_log.value += bubble('Running AI water detection... (can take a minute)', 'bot')
    try:
        water, bounds, km2 = detect_water_ai(box)
        for layer in list(m.layers):            # remove the previous AI result, if any
            if layer.name == 'Water (AI)':
                m.remove_layer(layer)
        m.add(water_overlay(water, bounds))
        chat_log.value += bubble(f'AI found {km2:.2f} km² of water in your box.', 'bot')
    except Exception as e:
        chat_log.value += bubble(f'Could not analyse the box: {e}', 'bot')

btn_box = widgets.Button(description='AI: water in drawn box', icon='square-o')
btn_box.add_class('pill')
btn_box.on_click(water_in_box)

# Placeholder for the classification step (final delivery)
btn_lp = widgets.Button(description='Lakes & ponds (coming)', disabled=True)
btn_lp.add_class('pill')

buttons = widgets.HBox(btns + [btn_box, btn_lp], layout=widgets.Layout(margin='12px 0 0 0'))

# --- Assemble: chat | map, buttons underneath ---
app = widgets.VBox([css, widgets.HBox([chat_panel, map_area]), buttons])
app.add_class('app')
app

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.

TraitError: The 'east' trait of a Map instance expected a float, not the NoneType None.